# 04c · ¿Por qué cada municipio tiene la predicción que tiene?

> **Reto 2 del INE:** "¿Qué zonas podrían presentar mayor abstención en el futuro si no se interviene?"

**A qué responde este paso.** El notebook 04 da, para cada municipio, cuántos puntos se prevé que quede arriba o abajo del país en 2027. Quien consulta el mapa pregunta enseguida **por qué**. Aquí repartimos esa cifra entre los cinco grupos de variables del modelo, para que la ficha del municipio pueda decir, por ejemplo: "su historia lo baja 12 puntos; el calendario lo sube 2".

**Decisión de Sofía Ordaz del 22 de octubre de 2026 (D29):** mostrar en el mapa cuánto aporta cada grupo de variables a la predicción de cada municipio, medido frente al país.

## Resumen de lo que encontramos

1. **El reparto cuadra:** para los 2,477 municipios, los cinco aportes suman exactamente los puntos frente al país (diferencia menor a 1e-13) y el modelo reentrenado da la misma predicción que el notebook 04.
2. **En los 495 rojos, lo que más baja la predicción es el calendario en 332 municipios y la historia en 163.** Ningún otro grupo es el principal en ningún rojo.
3. **Las dos historias del rojo se ven en el reparto** (medianas en puntos frente al país):

   | Rojos | Municipios | Historia | Calendario y usos y costumbres | Condiciones sociales |
   |---|---:|---:|---:|---:|
   | Sin elección municipal el 6 de junio | 413 | −6.8 | **−13.8** | +9.2 |
   | Con elección municipal | 82 | **−11.5** | +1.9 | −0.8 |

   - En los que **no eligen ayuntamiento**, el calendario los baja casi 14 puntos. Sus condiciones sociales (municipios chicos y rurales, que suelen votar más) los **suben** 9; no alcanza.
   - En los que **sí eligen ayuntamiento** (Tijuana, Juárez, Mexicali…), lo que los baja es su **historia**: votan poco siempre.
4. **Lista y población y competencia política casi no mueven nada** (menos de 1 punto en el municipio típico), igual que en la importancia por permutación del notebook 04b.
5. **Las dos partes del modelo reparten casi igual la historia y el calendario** (correlación de 0.95 y 0.98 entre municipios); las condiciones sociales, menos (0.65). Es el grupo más incierto del reparto.
6. **Archivo:** `data/processed/aportes_2027.parquet`, que el notebook 05 agrega a `municipios.csv` en `resultados-1.5` como `aporte_historia`, `aporte_calendario`, `aporte_lista`, `aporte_social` y `aporte_politica`.

## Palabras que usamos

| Palabra | Qué significa |
|---|---|
| **Aporte** | Cuántos puntos sube (+) o baja (−) la predicción de un municipio un grupo de variables, frente al país. Los cinco aportes suman los "puntos frente al país" del municipio. |
| **Grupo (bloque)** | Los cinco grupos de variables del modelo: historia, calendario y usos y costumbres, lista y población, condiciones sociales y competencia política (notebook 04). |
| **Frente al país** | Participación prevista del municipio menos la del país (51.4 % en 2027), en puntos. |

## 0 · El modelo final, igual que en el notebook 04

Volvemos a entrenar el modelo final con los mismos datos, variables y semillas que el notebook 04 (ensamble de Huber y random forest, D21) y comprobamos que da **exactamente** la misma desviación prevista que guardó `riesgo_2027.parquet`.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import polars as pl
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import HuberRegressor
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings('ignore')
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_tbl_width_chars(200)

raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file() and (p / 'data').is_dir())
P = raiz / 'data/processed'
faltan = [f for f in ['variables_transiciones.parquet', 'riesgo_2027.parquet'] if not (P / f).exists()]
assert not faltan, f'Primero corre los notebooks 03b y 04; faltan {faltan}'
variables = pl.read_parquet(P / 'variables_transiciones.parquet')
riesgo = pl.read_parquet(P / 'riesgo_2027.parquet')

# Las mismas listas del notebook 04
HISTORIA = ['desviacion_presidencial', 'desviacion_intermedia_previa', 'vecinos_desviacion_presidencial']
LOCALES = ['gobernador_mismo_dia_intermedia', 'diputacion_local_mismo_dia_intermedia', 'ayuntamiento_mismo_dia_intermedia',
           'gobernador_mismo_dia_presidencial', 'diputacion_local_mismo_dia_presidencial', 'ayuntamiento_mismo_dia_presidencial', 'es_sni']
POLITICAS = ['margen_presidencial', 'partidos_efectivos_presidencial']
LISTA = ['pct_rural', 'pct_hombres', 'pct_20_29', 'pct_65_mas', 'crecimiento_lista', 'cobertura_lista_20mas']
SOCIALES = ['escolaridad_promedio', 'pct_habla_lengua_indigena', 'pct_viviendas_internet', 'pct_residia_otro_pais_hace5', 'pct_residia_otra_entidad_hace5',
            'poblacion_log', 'pct_pobreza', 'pct_pobreza_extrema', 'pct_carencia_salud', 'pct_rezago_educativo', 'marginacion', 'pct_localidades_menos_5000',
            'pct_viviendas_emigrantes', 'pct_viviendas_remesas', 'pct_viviendas_retorno', 'homicidios_100mil_log']
TODAS = HISTORIA + LOCALES + POLITICAS + LISTA + SOCIALES
BLOQUES = {'historia': HISTORIA, 'calendario': LOCALES, 'lista': LISTA, 'social': SOCIALES, 'politica': POLITICAS}
NOMBRES = {'historia': 'Historia de participación', 'calendario': 'Calendario y usos y costumbres', 'lista': 'Lista y población',
           'social': 'Condiciones sociales', 'politica': 'Competencia política'}

def con_resultado(T):
    return variables.filter((pl.col('eleccion_objetivo') == T) & pl.col('desviacion_objetivo').is_not_null()
                            & ~pl.col('sin_casilla_objetivo').fill_null(False) & pl.col('desviacion_presidencial').is_not_null())

juntos = pl.concat([con_resultado(2015), con_resultado(2021)], how='diagonal_relaxed')
X, y = juntos.select(TODAS).to_numpy().astype(float), juntos['desviacion_objetivo'].to_numpy()
huber = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), HuberRegressor(alpha=1.0, epsilon=1.35, max_iter=3000)).fit(X, y)
bosque = make_pipeline(SimpleImputer(strategy='median'),
                       RandomForestRegressor(n_estimators=300, min_samples_leaf=5, max_features=0.6, random_state=0, n_jobs=-1)).fit(X, y)

futuro = variables.filter(pl.col('eleccion_objetivo') == 2027)
X_futuro = futuro.select(TODAS).to_numpy().astype(float)
prevista = (huber.predict(X_futuro) + bosque.predict(X_futuro)) / 2
compara = futuro.select('cvegeo').with_columns(nueva=pl.Series(prevista)).join(riesgo.select('cvegeo', 'desviacion_prevista'), on='cvegeo')
diferencia = (compara['nueva'] - compara['desviacion_prevista']).abs().max()
assert compara.height == riesgo.height and diferencia < 1e-9, diferencia
print(f'{compara.height:,} municipios · diferencia máxima con el notebook 04: {diferencia:.1e} puntos')

2,477 municipios · diferencia máxima con el notebook 04: 1.4e-14 puntos


## 1 · Cómo se reparte una predicción

El modelo es el promedio de dos partes, y cada una se reparte de forma distinta:

- **Huber (lineal):** la predicción es una constante más la suma de *peso × valor estandarizado* de cada variable. El reparto es exacto: cada variable aporta su término.
- **Random forest:** cada árbol lleva al municipio de la raíz a una hoja, pasando por varias preguntas ("¿la desviación presidencial es menor a −5?"). En cada pregunta, la predicción del árbol cambia; ese cambio se le atribuye a la variable de la pregunta. Al final, la hoja es el promedio de la raíz más todos los cambios. Se promedian los 300 árboles. Es el método de "contribuciones por camino" (Saabas), que también reparte la predicción completa.

El aporte de cada variable en el ensamble es el promedio de sus dos aportes. **Comprobación:** constante + suma de aportes debe dar la predicción, en cada parte y en el ensamble.

In [2]:
# Huber: aporte exacto de cada variable (peso × valor estandarizado)
Z = huber[1].transform(huber[0].transform(X_futuro))
aporte_huber, base_huber = Z * huber[2].coef_, huber[2].intercept_

# Random forest: contribuciones por camino de cada árbol (Saabas), promediadas
X_imp, rf = bosque[0].transform(X_futuro), bosque[1]
aporte_rf, base_rf = np.zeros_like(X_imp), 0.0
for arbol in rf.estimators_:
    t = arbol.tree_
    valor = t.value[:, 0, 0]
    camino = arbol.decision_path(X_imp).tocsr()
    base_rf += valor[0]
    for i in range(X_imp.shape[0]):
        nodos = np.sort(camino.indices[camino.indptr[i]:camino.indptr[i + 1]])  # en sklearn, cada hijo tiene un número mayor que su padre
        np.add.at(aporte_rf[i], t.feature[nodos[:-1]], valor[nodos[1:]] - valor[nodos[:-1]])
aporte_rf, base_rf = aporte_rf / len(rf.estimators_), base_rf / len(rf.estimators_)

aporte = (aporte_huber + aporte_rf) / 2
base = (base_huber + base_rf) / 2
for nombre, b, a, p in [('Huber', base_huber, aporte_huber, huber.predict(X_futuro)), ('Random forest', base_rf, aporte_rf, bosque.predict(X_futuro)),
                        ('Ensamble', base, aporte, prevista)]:
    error = np.abs(b + a.sum(axis=1) - p).max()
    assert error < 1e-9, (nombre, error)
    print(f'{nombre:14} constante = {b:5.2f} pp · constante + aportes = predicción (diferencia máxima {error:.1e})')

Huber          constante =  7.46 pp · constante + aportes = predicción (diferencia máxima 1.8e-14)
Random forest  constante =  7.37 pp · constante + aportes = predicción (diferencia máxima 8.5e-14)
Ensamble       constante =  7.41 pp · constante + aportes = predicción (diferencia máxima 4.3e-14)


**Lectura:** el reparto cuadra en las dos partes. La constante (unos 7.4 puntos) es la desviación del municipio típico con el que aprendió el modelo: los municipios chicos votan más que el país, y en el entrenamiento todos pesan igual.

Una constante así no le dice nada a quien consulta el mapa. Por eso, en la sección 2 medimos cada aporte **frente al país**.

## 2 · Cinco grupos, medidos frente al país

1. **Sumamos los aportes por grupo.** Con 34 variables que a veces dicen lo mismo (por ejemplo, pobreza y rezago educativo), el reparto entre variables sueltas no es confiable; el de grupos lo es más.
2. **Centramos cada grupo en el país:** a cada aporte le restamos su promedio ponderado por la lista de 2024. Así, un aporte de 0 significa "en este grupo, el municipio se parece al país".

**Por qué así:** en el notebook 05, la participación de 2027 = nivel del país + desviación prevista + un ajuste común que hace que el promedio ponderado por la lista sea el nivel del país. Por eso, "puntos frente al país" = desviación prevista − su promedio ponderado. Con los grupos centrados, **los cinco aportes suman exactamente los puntos frente al país** que muestra la ficha. Lo comprobamos.

In [3]:
lista = riesgo.select('cvegeo', 'lista_2024')
grupos = futuro.select('cvegeo').with_columns([pl.Series(f'bruto_{g}', aporte[:, [TODAS.index(v) for v in vs]].sum(axis=1)) for g, vs in BLOQUES.items()])
grupos = grupos.with_columns(desviacion_prevista=pl.Series(prevista)).join(lista, on='cvegeo', how='left')
w = grupos['lista_2024'].fill_null(0).cast(pl.Float64)
centros = {g: float((grupos[f'bruto_{g}'] * w).sum() / w.sum()) for g in BLOQUES}
aportes = grupos.with_columns([(pl.col(f'bruto_{g}') - centros[g]).alias(f'aporte_{g}') for g in BLOQUES])
frente_al_pais = aportes['desviacion_prevista'] - float((aportes['desviacion_prevista'] * w).sum() / w.sum())
error = (aportes.select(pl.sum_horizontal([f'aporte_{g}' for g in BLOQUES]))[:, 0] - frente_al_pais).abs().max()
assert error < 1e-9, error
print(f'Los cinco aportes suman los puntos frente al país (diferencia máxima {error:.1e})')
print('Promedio de cada grupo antes de centrar, ponderado por la lista (pp):', {NOMBRES[g]: round(c, 2) for g, c in centros.items()})
aportes = aportes.with_columns(frente_al_pais=frente_al_pais)

Los cinco aportes suman los puntos frente al país (diferencia máxima 3.6e-14)
Promedio de cada grupo antes de centrar, ponderado por la lista (pp): {'Historia de participación': -3.47, 'Calendario y usos y costumbres': 5.04, 'Lista y población': 0.02, 'Condiciones sociales': -9.2, 'Competencia política': -0.15}


## 3 · Qué grupo pesa más

Medimos el tamaño típico de cada aporte (mediana del valor absoluto) en todos los municipios y en los rojos, y qué grupo es el que **más baja** la predicción de cada municipio en rojo.

In [4]:
semaforo = (riesgo.select('cvegeo', 'percentil_riesgo', 'ayuntamiento_mismo_dia_2027', 'es_sni')
            .with_columns(rojo=(100 - pl.col('percentil_riesgo')) >= 80))
a = aportes.join(semaforo, on='cvegeo')
G = list(BLOQUES)
tamano = pl.DataFrame({
    'Grupo': [NOMBRES[g] for g in G],
    'Tamaño típico, todos (pp)': [round(a[f'aporte_{g}'].abs().median(), 1) for g in G],
    'Tamaño típico, rojos (pp)': [round(a.filter('rojo')[f'aporte_{g}'].abs().median(), 1) for g in G],
    'Mediana en los rojos (pp)': [round(a.filter('rojo')[f'aporte_{g}'].median(), 1) for g in G],
})
print(tamano)
rojos = a.filter('rojo')
mas_baja = rojos.select([f'aporte_{g}' for g in G]).to_numpy().argmin(axis=1)
print(f'\nRojos: {rojos.height}. Grupo que más baja su predicción:')
print(pl.DataFrame({'Grupo': [NOMBRES[G[i]] for i in mas_baja]}).group_by('Grupo').len().sort('len', descending=True))
print('\nMediana de cada aporte en los rojos, según si eligen ayuntamiento el 6 de junio de 2027 (pp):')
print(rojos.group_by('ayuntamiento_mismo_dia_2027').agg(pl.len().alias('Municipios'), *[pl.col(f'aporte_{g}').median().round(1).alias(NOMBRES[g]) for g in G])
      .sort('ayuntamiento_mismo_dia_2027'))

shape: (5, 4)
┌────────────────────────────────┬───────────────────────────┬───────────────────────────┬───────────────────────────┐
│ Grupo                          ┆ Tamaño típico, todos (pp) ┆ Tamaño típico, rojos (pp) ┆ Mediana en los rojos (pp) │
│ ---                            ┆ ---                       ┆ ---                       ┆ ---                       │
│ str                            ┆ f64                       ┆ f64                       ┆ f64                       │
╞════════════════════════════════╪═══════════════════════════╪═══════════════════════════╪═══════════════════════════╡
│ Historia de participación      ┆ 6.7                       ┆ 8.0                       ┆ -8.0                      │
│ Calendario y usos y costumbres ┆ 1.0                       ┆ 13.8                      ┆ -13.8                     │
│ Lista y población              ┆ 0.5                       ┆ 0.7                       ┆ -0.2                      │
│ Condiciones sociales           ┆

**¿Coinciden las dos partes del modelo?** Si Huber y el random forest reparten de forma muy distinta, el reparto sería frágil. Comparamos, grupo por grupo, el aporte de cada parte (correlación entre municipios).

In [5]:
coincidencia = []
for g, vs in BLOQUES.items():
    idx = [TODAS.index(v) for v in vs]
    h, r = aporte_huber[:, idx].sum(axis=1), aporte_rf[:, idx].sum(axis=1)
    coincidencia.append({'Grupo': NOMBRES[g], 'Correlación Huber vs random forest': round(float(np.corrcoef(h, r)[0, 1]), 2),
                         'Tamaño típico Huber (pp)': round(float(np.median(np.abs(h - np.average(h, weights=w)))), 1),
                         'Tamaño típico random forest (pp)': round(float(np.median(np.abs(r - np.average(r, weights=w)))), 1)})
print(pl.DataFrame(coincidencia))

shape: (5, 4)
┌────────────────────────────────┬────────────────────────────────────┬──────────────────────────┬──────────────────────────────────┐
│ Grupo                          ┆ Correlación Huber vs random forest ┆ Tamaño típico Huber (pp) ┆ Tamaño típico random forest (pp) │
│ ---                            ┆ ---                                ┆ ---                      ┆ ---                              │
│ str                            ┆ f64                                ┆ f64                      ┆ f64                              │
╞════════════════════════════════╪════════════════════════════════════╪══════════════════════════╪══════════════════════════════════╡
│ Historia de participación      ┆ 0.95                               ┆ 6.6                      ┆ 6.9                              │
│ Calendario y usos y costumbres ┆ 0.98                               ┆ 1.6                      ┆ 1.3                              │
│ Lista y población              ┆ 0.47         

## 4 · Ejemplos

Cuatro municipios de distinto tipo: una ciudad del norte que elige ayuntamiento y vota poco siempre (Tijuana), una capital sin elección municipal en 2027 (Xalapa), un municipio con participación muy baja y repetida (Cherán) y una ciudad cerca del promedio (Guadalajara).

In [6]:
ejemplos = {'02004': 'Tijuana', '30087': 'Xalapa', '16024': 'Cherán', '14039': 'Guadalajara'}
tabla = (aportes.filter(pl.col('cvegeo').is_in(list(ejemplos))).with_columns(Municipio=pl.col('cvegeo').replace_strict(ejemplos))
         .select('Municipio', *[pl.col(f'aporte_{g}').round(1).alias(NOMBRES[g]) for g in G], pl.col('frente_al_pais').round(1).alias('Frente al país')))
print(tabla)

shape: (4, 7)
┌─────────────┬───────────────────────────┬────────────────────────────────┬───────────────────┬──────────────────────┬──────────────────────┬────────────────┐
│ Municipio   ┆ Historia de participación ┆ Calendario y usos y costumbres ┆ Lista y población ┆ Condiciones sociales ┆ Competencia política ┆ Frente al país │
│ ---         ┆ ---                       ┆ ---                            ┆ ---               ┆ ---                  ┆ ---                  ┆ ---            │
│ str         ┆ f64                       ┆ f64                            ┆ f64               ┆ f64                  ┆ f64                  ┆ f64            │
╞═════════════╪═══════════════════════════╪════════════════════════════════╪═══════════════════╪══════════════════════╪══════════════════════╪════════════════╡
│ Tijuana     ┆ -12.0                     ┆ 1.9                            ┆ -0.0              ┆ -4.5                 ┆ -0.3                 ┆ -14.9          │
│ Guadalajara ┆ 0.2       

**Lectura de los ejemplos:**

- **Tijuana (−14.9):** su historia la baja 12 puntos (votó menos que el país en las seis elecciones) y sus condiciones sociales 4.5; el calendario la sube 1.9 porque sí elige ayuntamiento y gubernatura.
- **Xalapa (−8.5):** casi todo es el calendario (−11.7): Veracruz no elige ayuntamiento en 2027. Su historia la sube un poco (+3.2). Es el caso que se revisó en D25 y D27.
- **Cherán (−36.0):** su historia de participación muy baja (−30.8) y que elige por usos y costumbres (−11.3).
- **Guadalajara (−2.2):** cerca del país; ningún grupo pesa mucho.

## 5 · Advertencias y recomendación

- **Es un reparto de la predicción, no una explicación causal.** Dice qué variables empujan la cifra del modelo, no por qué la gente vota menos.
- **El reparto del random forest es una aproximación:** las contribuciones por camino tienden a darle más peso a las variables de las primeras preguntas de cada árbol. Por eso lo promediamos con Huber, cuyo reparto es exacto, y lo mostramos por grupos.
- **Las condiciones sociales son el grupo más incierto:** Huber y el random forest coinciden menos en él (correlación de 0.65).
- **Algunos grupos se parecen entre sí:** por ejemplo, hablar una lengua indígena (condiciones sociales) y elegir por usos y costumbres (calendario). Parte del peso de uno puede aparecer en el otro.
- **Se mide frente al país (ponderado por personas):** un aporte de 0 es "como el país", no "no importa".

**Recomendación:** mostrar los cinco aportes en la ficha del municipio, con el título "¿Qué empuja esta predicción?" y la nota "Reparto de la predicción del modelo; no son causas probadas". Junto a las barras, los hechos del municipio que ya están en los datos (elecciones por debajo del país, si elige ayuntamiento, usos y costumbres).

In [7]:
salida = aportes.select('cvegeo', *[pl.col(f'aporte_{g}').round(2) for g in G], pl.col('frente_al_pais').round(2)).sort('cvegeo')
salida.write_parquet(P / 'aportes_2027.parquet')
print(f'aportes_2027.parquet: {salida.height:,} municipios · columnas: {salida.columns}')

aportes_2027.parquet: 2,477 municipios · columnas: ['cvegeo', 'aporte_historia', 'aporte_calendario', 'aporte_lista', 'aporte_social', 'aporte_politica', 'frente_al_pais']
